# Notebook 3 — Egyptian Data Sources & Standardisation

**Project:** Robson Classification and C-Section Predictive System
**Stage:** Phase 2 — Egyptian comparison layer
**Scripts covered (8):**

| Script | Role |
|--------|------|
| `dhs_egypt.py` | Read Egypt DHS 2014 + EHIS 2015 microdata |
| `ehis_2015.py` | Resolve the 43.8% vs 51.8% contradiction |
| `stage3_confounding.py` | Direct standardisation method + US Group 1 test |
| `stage4_setting_and_wider_adjustment.py` | Expanded confounding control (6 variables) |
| `egypt_standardisation.py` | Apply the same method to Egyptian DHS data |
| `egypt_gap_v2.py` | **The main result** — indirect standardisation on 8 categories |
| `egypt_cost.py` | Economic cost model |
| `egypt_governorates.py` | Governorate-level data (descriptive) |

---

### Two different standardisation methods in this notebook

This notebook contains two distinct analyses that both use "standardisation" but for different purposes:

| | Direct Standardisation | Indirect Standardisation |
|---|---|---|
| **Question** | Does the payer gap in CS rate survive adjustment for case-mix? | How much of Egypt's CS rate exceeds what the reference would predict? |
| **Method** | Reweight stratum-specific rates to a common population profile | Multiply Egypt's group sizes by reference group-specific rates |
| **Scripts** | `stage3_confounding.py`, `stage4_...py`, `egypt_standardisation.py` | `egypt_gap_v2.py` |
| **Key result** | US payer gap disappears after adjustment; Egypt's private-public gap shrinks but survives | Benha +4.46pp, Qena +20.65pp |

### The corrected numbers

This notebook documents the transition from the **old 10-category** standardisation to the **corrected 8-category** method:

| | Old (10-cat) | **Corrected (8-cat)** | Why it changed |
|---|---|---|---|
| Benha Gap A | +7.54 pp | **+4.46 pp** | Onset-of-labour groups merged |
| Qena Gap A | +20.69 pp | **+20.65 pp** | Same correction, smaller effect |
| Brier score | 0.1857 | **0.0886** | class-weighting removed (Notebook 4) |

---

## Part 1 — Egyptian Data Sources

### 1.1 — `dhs_egypt.py`: Reading DHS 2014 and EHIS 2015

**What DHS can and cannot give us:**

Robson needs six criteria. DHS carries three:

| Criterion | Available? | Source |
|-----------|-----------|--------|
| Parity | ✓ | Birth order (`bord`) |
| Previous caesarean | ✓ (within recall window) | Mode of delivery (`m17`) |
| Number of fetuses | ✓ | Multiple-birth code (`b0`) |
| Fetal presentation | ✗ | Not in DHS |
| Onset of labour | ✗ | Not in DHS |
| Gestational age | ✗ | Only crude size-at-birth |

**So DHS cannot produce a Robson table.** But it CAN give a nationally representative parity and previous-caesarean structure — which is exactly what the two hospital studies cannot give (they are tertiary referral centres).

**Key technical points:**
- DHS is one row per WOMAN, with birth history variables suffixed `_1` to `_6`. The script reshapes to one row per BIRTH.
- Weights (`v005 / 1,000,000`) are mandatory — DHS uses stratified cluster sampling.
- The recall window matters: `m17` is only asked for births in the 5 years before the survey.

In [ ]:
"""
=============================================================================
EGYPT DHS — first pass
=============================================================================
Put the two downloaded ZIPs (or their extracted .DTA files) in this folder
and run:

    pip install pandas pyreadstat
    python dhs_egypt.py

WHAT DHS CAN AND CANNOT GIVE US

Robson needs six criteria. DHS carries three of them:

    parity ................ yes, from birth order
    previous caesarean .... yes, but only within the recall window
    number of fetuses ..... yes, from the multiple-birth code
    fetal presentation .... NO
    onset of labour ....... NO
    gestational age ....... NO (only a crude size-at-birth report)

So DHS cannot produce a Robson table. What it CAN do is give a nationally
representative parity and previous-caesarean structure — which is exactly
what the two hospital studies cannot give, because they are tertiary
referral centres. The two sources are complements, not substitutes.

THE RECALL WINDOW MATTERS
Mode of delivery (m17) is asked only for births in the five years before
the survey. For a woman whose earlier births fall outside that window, we
cannot know whether they were caesareans — so "previous caesarean" is only
reliably derivable for women whose whole birth history sits inside it.
The script reports how large that usable subset is instead of hiding it.

WEIGHTS ARE NOT OPTIONAL
DHS uses a stratified cluster design. An unweighted percentage is not the
national figure. v005 is divided by 1,000,000 to give the weight.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import zipfile, io

pd.set_option("display.width", 200)


def find_dta(stem_hint):
    """Locate a .DTA on disk, or pull it straight out of its ZIP."""
    for f in Path(".").glob("*.DTA"):
        if stem_hint.lower() in f.name.lower():
            return f
    for f in Path(".").glob("*.dta"):
        if stem_hint.lower() in f.name.lower():
            return f
    for z in list(Path(".").glob("*.ZIP")) + list(Path(".").glob("*.zip")):
        if stem_hint.lower() not in z.name.lower():
            continue
        with zipfile.ZipFile(z) as zf:
            for name in zf.namelist():
                if name.upper().endswith(".DTA"):
                    print(f"  extracting {name} from {z.name}")
                    zf.extract(name, ".")
                    return Path(name)
    return None


def load(stem_hint, label):
    p = find_dta(stem_hint)
    if p is None:
        print(f"[skip] {label}: no .DTA or .ZIP matching '{stem_hint}' in this folder")
        return None
    print(f"Loading {label}: {p}")
    df = pd.read_stata(p, convert_categoricals=False)
    print(f"  {len(df):,} women, {len(df.columns):,} variables\n")
    return df


def births_long(df, label, max_births=6):
    """
    The Individual Recode is one row per WOMAN, with birth-history variables
    suffixed _1 .. _6 (most recent first). Reshape to one row per BIRTH.
    """
    keep_woman = [c for c in ["caseid","v005","v012","v024","v025","v106",
                              "v190","v201","v208","v212"] if c in df.columns]
    rows = []
    for i in range(1, max_births+1):
        cols = {}
        for base in ["m17","m15","bord","b0","b19","b5","m19","m18"]:
            c = f"{base}_{i}"
            if c in df.columns:
                cols[c] = base
        if "m17" not in cols.values():
            continue
        sub = df[keep_woman + list(cols)].rename(columns=cols).copy()
        sub["birth_index"] = i
        rows.append(sub)
    if not rows:
        print(f"[{label}] no m17_* variables found — check the file")
        return None
    b = pd.concat(rows, ignore_index=True)
    b = b[b["m17"].notna()]
    b["weight"] = b["v005"] / 1_000_000 if "v005" in b else 1.0
    b["cs"] = (pd.to_numeric(b["m17"], errors="coerce") == 1).astype(float)
    print(f"[{label}] {len(b):,} births with a recorded mode of delivery")
    return b


def wpct(d, col="cs"):
    w = d["weight"]
    return 100 * (d[col] * w).sum() / w.sum() if w.sum() else np.nan


# =============================================================================
ed14 = load("IR61", "EDHS 2014")
if ed14 is None:
    ed14 = load("EGIR", "EDHS 2014")
eh15 = load("IQ73", "EHIS 2015")
if eh15 is None:
    eh15 = load("EGIQ", "EHIS 2015")

results = {}
for df, label in [(ed14, "EDHS 2014"), (eh15, "EHIS 2015")]:
    if df is None:
        continue
    b = births_long(df, label)
    if b is None:
        continue
    results[label] = b

    print("=" * 70)
    print(f"{label}")
    print("=" * 70)
    print(f"  Caesarean rate, ALL births in the recall window")
    print(f"    weighted   : {wpct(b):.2f}%")
    print(f"    unweighted : {100*b['cs'].mean():.2f}%   <- not the national figure")

    last = b[b["birth_index"] == 1]
    print(f"  Caesarean rate, MOST RECENT birth only")
    print(f"    weighted   : {wpct(last):.2f}%   (n = {len(last):,})")
    print("""
    Published figures usually refer to one of these two definitions.
    If they differ here, that alone may explain part of the gap between
    the 51.8% and 43.8% national figures.
""")

    # --- parity -----------------------------------------------------------
    if "bord" in b.columns:
        bo = pd.to_numeric(b["bord"], errors="coerce")
        b["parity_group"] = np.where(bo == 1, "nulliparous (1st birth)",
                             np.where(bo.notna(), "multiparous", None))
        t = b.dropna(subset=["parity_group"]).groupby("parity_group").apply(
            lambda x: pd.Series({"n_births": len(x),
                                 "share_pct": round(100*x["weight"].sum()/b["weight"].sum(),2),
                                 "cs_rate_pct": round(wpct(x),2)}))
        print("  By parity at that birth (Robson's first criterion):")
        print(t.to_string(), "\n")

    # --- plurality --------------------------------------------------------
    if "b0" in b.columns:
        b["multiple"] = pd.to_numeric(b["b0"], errors="coerce").fillna(0) > 0
        print(f"  Multiple births: {100*b['multiple'].mean():.2f}% "
              f"| caesarean rate {wpct(b[b['multiple']]):.2f}% "
              f"vs singleton {wpct(b[~b['multiple']]):.2f}%\n")

    # --- previous caesarean, within the window ----------------------------
    b = b.sort_values(["caseid", "birth_index"], ascending=[True, False])
    b["prev_cs_in_window"] = (b.groupby("caseid")["cs"].cumsum() - b["cs"]) > 0
    if "v201" in b.columns and "bord" in b.columns:
        # usable only where every earlier birth is also observed
        b["history_complete"] = pd.to_numeric(b["bord"], errors="coerce") <= \
                                b.groupby("caseid")["birth_index"].transform("max")
        usable = b[b["history_complete"]]
        print(f"  Previous caesarean is derivable for {len(usable):,} of {len(b):,} births "
              f"({100*len(usable)/len(b):.1f}%)")
        if len(usable):
            print(f"    with a previous caesarean : {100*usable['prev_cs_in_window'].mean():.2f}% "
                  f"| caesarean rate {wpct(usable[usable.prev_cs_in_window]):.2f}%")
            print(f"    without                   : caesarean rate "
                  f"{wpct(usable[~usable.prev_cs_in_window]):.2f}%")
        print()

    # --- place of delivery, the public/private divide ---------------------
    if "m15" in b.columns:
        m15 = pd.to_numeric(b["m15"], errors="coerce")
        b["place"] = np.select(
            [m15.between(11,12), m15.between(21,26), m15.between(31,36)],
            ["home", "public facility", "private facility"], default="other/unknown")
        t = b.groupby("place").apply(lambda x: pd.Series({
            "share_pct": round(100*x["weight"].sum()/b["weight"].sum(),2),
            "cs_rate_pct": round(wpct(x),2)}))
        print("  By place of delivery:")
        print(t.to_string(), "\n")

    # --- residence and wealth --------------------------------------------
    for v, name in [("v025","urban/rural"), ("v190","wealth quintile"), ("v024","region")]:
        if v in b.columns:
            t = b.groupby(v).apply(lambda x: pd.Series({
                "share_pct": round(100*x["weight"].sum()/b["weight"].sum(),2),
                "cs_rate_pct": round(wpct(x),2)}))
            print(f"  By {name} (codes — see the DHS recode manual):")
            print(t.to_string(), "\n")

    b.to_csv(f"dhs_{label.replace(' ','_').lower()}_births.csv", index=False)
    print(f"  saved dhs_{label.replace(' ','_').lower()}_births.csv\n")

# --- the headline comparison ---------------------------------------------
if len(results) == 2:
    print("=" * 70)
    print("THE 2014 vs 2015 DISCREPANCY")
    print("=" * 70)
    for label, b in results.items():
        last = b[b["birth_index"] == 1]
        print(f"  {label:<12} all births {wpct(b):>6.2f}%   most recent only {wpct(last):>6.2f}%")
    print("""
  Published: EDHS 2014 = 51.8%, EHIS 2015 = 43.8% — eight points apart,
  one year apart. Candidate explanations to test from here:
    - a different definition (all births in the window vs most recent)
    - a different recall window (five years vs three, or vs two)
    - a different age range of women
    - EHIS being a smaller survey with a different sample design
  If the difference reproduces under one definition and vanishes under
  another, that is the answer — and nobody has published it.
""")

### 1.2 — `ehis_2015.py`: Resolving the 43.8% vs 51.8% Contradiction

**The problem:** Two national surveys, one year apart, with an 8-point gap:
- EDHS 2014 = 51.8%
- EHIS 2015 = 43.8%

**The answer this script found:** They measure different things.

| Survey | What it measures | Denominator |
|--------|-----------------|-------------|
| EDHS 2014 (51.8%) | CS rate among **births** in the last 5 years | Births |
| EHIS 2015 (43.8%) | Share of **mothers** who ever had a CS | Women |

EHIS has no birth history — it asks total children ever born (`a609`) and how many were caesareans (`a625`). That's a **lifetime** measure. Because the CS rate has risen steeply, a lifetime measure carries older, lower-rate births and is dragged down.

**This distinction has not been published.** The two figures are routinely quoted side-by-side as though comparable.

The script also extracts national attitude questions — whether women believe vaginal birth is possible after a CS, when the decision was taken, and stated reasons.

In [ ]:
"""
=============================================================================
EGYPT HEALTH ISSUES SURVEY 2015 — what it actually contains
=============================================================================
    python ehis_2015.py

TWO JOBS

1. Test why 43.8% and 51.8% disagree.
   EDHS 2014 measures caesareans among births in the five years before the
   survey. EHIS 2015 has no birth history at all — it asks total children
   ever born (a609) and how many of those were caesareans (a625). That is a
   LIFETIME measure. Because the caesarean rate has risen steeply, a
   lifetime measure carries older, lower-rate births and is dragged down.
   If sum(a625)/sum(a609) lands near 43.8%, the discrepancy is a difference
   of definition, not a contradiction — and that is worth stating, because
   the two figures are quoted side by side as though comparable.

2. Extract the national attitude and decision-timing questions.
   These are the demand side, on a nationally representative sample:
   whether a woman believes vaginal birth is possible after a caesarean,
   when the decision was taken, and the stated reasons.
=============================================================================
"""

import pandas as pd
import numpy as np

FILE = "EGIQ73FL.DTA"
print(f"Loading {FILE} ...")
raw = pd.read_stata(FILE, convert_categoricals=False)
lab = pd.read_stata(FILE, convert_categoricals=False, iterator=True).variable_labels()
print(f"  {len(raw):,} women, {len(raw.columns):,} variables\n")

# ---- weight ---------------------------------------------------------------
wcol = next((c for c in ["v005","awfactt","a005","weight"] if c in raw.columns), None)
if wcol:
    raw["w"] = pd.to_numeric(raw[wcol], errors="coerce") / 1_000_000
    if raw["w"].mean() < 0.001 or raw["w"].mean() > 100:
        raw["w"] = pd.to_numeric(raw[wcol], errors="coerce")
    print(f"Weight variable: {wcol} (mean {raw['w'].mean():.3f})\n")
else:
    raw["w"] = 1.0
    print("WARNING: no weight variable found — results are unweighted\n")

num = lambda c: pd.to_numeric(raw[c], errors="coerce") if c in raw.columns else None

# ---- 1. the lifetime caesarean measure ------------------------------------
births = num("a609")      # total children ever born
cs     = num("a625")      # of those, how many by caesarean
print("=" * 68)
print("1. LIFETIME CAESAREAN MEASURE")
print("=" * 68)
if births is None or cs is None:
    print("  a609 or a625 missing — check the variable names")
else:
    # women who have given birth; treat missing caesarean count as zero only
    # where the woman was asked and answered nothing meaningful
    d = raw.assign(births=births, cs=cs)
    d = d[d["births"].notna() & (d["births"] > 0) & (d["births"] < 30)]
    d["cs"] = d["cs"].where(d["cs"].between(0, 30), np.nan)
    known = d[d["cs"].notna()]

    tot_b = (known["births"] * known["w"]).sum()
    tot_c = (known["cs"]     * known["w"]).sum()
    print(f"  Women who have given birth, with a usable caesarean count: {len(known):,}")
    print(f"  Weighted total births      : {tot_b:,.0f}")
    print(f"  Weighted total caesareans  : {tot_c:,.0f}")
    print(f"  LIFETIME caesarean rate    : {100*tot_c/tot_b:.2f}%")
    print(f"  Unweighted equivalent      : {100*known['cs'].sum()/known['births'].sum():.2f}%")
    print(f"\n  Published EHIS 2015 figure : 43.8%")
    print(f"  Published EDHS 2014 figure : 51.8%  (births in the last five years)")
    print("""
  If the lifetime figure lands near 43.8%, the two national numbers are
  measuring different things and should never be plotted on one trend line
  without saying so.
""")
    # proportion of mothers who have ever had a caesarean
    ever = (known["cs"] > 0)
    print(f"  Mothers who have EVER had a caesarean: "
          f"{100*(ever*known['w']).sum()/known['w'].sum():.2f}%")

# ---- 2. attitudes and decision timing -------------------------------------
print("\n" + "=" * 68)
print("2. NATIONAL ATTITUDES AND DECISION TIMING")
print("=" * 68)

def show(var):
    if var not in raw.columns:
        print(f"  [{var}] not present"); return
    s = pd.to_numeric(raw[var], errors="coerce")
    d = raw[s.notna()]
    if not len(d):
        print(f"  [{var}] no valid responses"); return
    t = (d.groupby(s[s.notna()]).apply(lambda x: 100*x["w"].sum()/d["w"].sum())
           .round(2).rename("pct").to_frame())
    t["n"] = d.groupby(s[s.notna()]).size()
    print(f"\n  {var} — {lab.get(var,'')}")
    print(t.to_string())

for v in ["a623", "a626", "a627", "a630", "a628", "a631",
          "a622a", "a622b", "a622c", "a622d", "a622e", "a621"]:
    show(v)

print("""

  Codes are numeric. To read them, open the survey's recode map, or
  re-read the file with convert_categoricals=True for these columns.

  a623 is the one to look at first. Benha's authors named a
  "once a caesarean always a caesarean" culture as a reason for a 100%
  caesarean rate in Robson group 5. This variable measures that belief
  on a national sample — the belief and the practice, from two
  independent Egyptian sources.

  a626 gives decision timing nationally, which is the counterpart to the
  Qena finding that 77% of births were pre-labour caesareans.
""")

# ---- 3. save --------------------------------------------------------------
cols = [c for c in ["caseid","w","a602","a609","a625","a621","a623","a626",
                    "a627","a628","a630","a631","a622a","a622b","a622c",
                    "a622d","a622e","a110","v025","v190","v024"]
        if c in raw.columns]
raw[cols].to_csv("ehis_2015_extract.csv", index=False)
print(f"Saved ehis_2015_extract.csv ({len(cols)} columns)")

---

## Part 2 — Direct Standardisation: Does the Payer Gap Survive Adjustment?

### 2.1 — `stage3_confounding.py`: The Method and the US Test

**Why this script exists:** Stage 2 invited the reader to compare safety outcomes ACROSS Robson groups. That invitation was wrong and is withdrawn here. Comparing Group 2a against Group 1 is confounded by indication — labour is induced FOR a reason.

**The valid comparison:** WITHIN one group, ACROSS practice settings. Group 1 fixes the clinical population (nulliparous, term, singleton, cephalic, spontaneous labour). Any variation left is variation in how labour was managed.

**Method — Direct Standardisation:**

Split Group 1 into strata (age × BMI × diabetes × hypertension). Compute each payer's CS rate within each stratum, then reweight to the whole group's risk profile:

```
standardised_rate = Σ (stratum_weight × payer_rate_in_stratum)
```

This answers: "If every payer served an identical mix of women, what would each one's CS rate be?"

**US Result:** Crude gap (Private 14.25% > Medicaid 11.97% > Self-pay 8.30%) → After standardisation: Medicaid **highest** (13.70%), Private 13.07%. The gap was age, not financial incentive.

In [ ]:
"""
=============================================================================
STAGE 3 — THE CLEAN TEST
Does a higher caesarean rate buy better outcomes?
=============================================================================

WHY THIS SCRIPT EXISTS

Stage 2 printed a line inviting the reader to compare safety outcomes ACROSS
Robson groups. That invitation was wrong and is withdrawn here.

Comparing group 2a against group 1 is confounded by indication. Labour is
induced FOR a reason — hypertension, diabetes, growth restriction,
post-dates. Worse neonatal outcomes in group 2a may reflect the indication
rather than the induction. The ARRIVE randomised trial found the opposite
for elective induction at 39 weeks, so observational data of this shape is
not evidence against induction. The same objection applies to comparing
group 1 against group 3: nulliparity is itself a risk factor.

Robson groups differ in exactly the characteristics that drive both
caesarean and outcome. Between-group causal claims are not available.

THE VALID COMPARISON is WITHIN one group, ACROSS practice settings. Robson
group 1 fixes the clinical population — nulliparous, term, singleton,
cephalic, spontaneous labour. Every woman arrived in labour on her own and
no caesarean in the group was scheduled. If one payer stratum operates at a
14.25% caesarean rate and another at 8.30%, and the babies do equally well,
the extra caesareans are not buying safety.

Residual confounding remains: payer correlates with maternal age, BMI and
comorbidity. So we do not compare crude rates. We standardise.

METHOD — DIRECT STANDARDISATION
Split group 1 into strata of maternal age x BMI x diabetes x hypertension.
Compute each payer's caesarean rate WITHIN each stratum, then reweight to
the risk profile of the whole group:

    standardised rate = SUM_over_strata ( w_s  x  rate_payer,s )
    where w_s = that stratum's share of all group 1 births

This answers: "if every payer served an identical mix of women, what would
each one's caesarean rate be?"

It is the same technique the project will use for Egypt in phase two, run
here on US data first as a rehearsal — and it is checkable, because unlike
the Egypt calculation we can see the true answer alongside it.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path

CACHE = "natality_2023_filtered.parquet"


# =============================================================================
# RISK STRATA
# =============================================================================

def build_strata(d):
    """Maternal age x BMI x any diabetes x any hypertensive disorder."""
    age = pd.to_numeric(d["MAGER"], errors="coerce")
    d["age_band"] = pd.cut(age, [0, 19, 24, 29, 34, 39, 99],
                           labels=["<20", "20-24", "25-29", "30-34", "35-39", "40+"])

    bmi = pd.to_numeric(d["BMI"], errors="coerce")
    bmi = bmi.where(bmi < 99)                      # 99.9 = unknown
    d["bmi_band"] = pd.cut(bmi, [0, 18.5, 25, 30, 35, 99],
                           labels=["<18.5", "18.5-24", "25-29", "30-34", "35+"])

    yes = {"Y"}
    d["diabetes"] = (d["RF_PDIAB"].astype(str).str.strip().isin(yes) |
                     d["RF_GDIAB"].astype(str).str.strip().isin(yes))
    d["hypertension"] = (d["RF_PHYPE"].astype(str).str.strip().isin(yes) |
                         d["RF_GHYPE"].astype(str).str.strip().isin(yes) |
                         d["RF_EHYPE"].astype(str).str.strip().isin(yes))

    d["stratum"] = (d["age_band"].astype(str) + "|" + d["bmi_band"].astype(str)
                    + "|" + d["diabetes"].astype(str) + "|" + d["hypertension"].astype(str))
    return d


def direct_standardise(d, group_col, outcome_col, min_cell=30):
    """
    Reweight each group's stratum-specific rate to the population risk profile.
    Strata where a group has fewer than min_cell births are dropped for that
    group and its weights renormalised, so a thin cell cannot swing the result.
    """
    valid = d[~d["stratum"].str.contains("nan")]
    weights = valid["stratum"].value_counts(normalize=True)

    out = {}
    for g, sub in valid.groupby(group_col, observed=True):
        rates = sub.groupby("stratum", observed=True)[outcome_col].agg(["mean", "size"])
        rates = rates[rates["size"] >= min_cell]
        if rates.empty:
            continue
        w = weights.reindex(rates.index).fillna(0)
        w = w / w.sum()
        out[g] = {
            "crude_%": 100 * sub[outcome_col].mean(),
            "standardised_%": 100 * float((w * rates["mean"]).sum()),
            "n": len(sub),
            "strata_used": len(rates),
            "coverage_%": 100 * float(weights.reindex(rates.index).fillna(0).sum()),
        }
    return pd.DataFrame(out).T


# =============================================================================
if __name__ == "__main__":
    if not Path(CACHE).exists():
        raise SystemExit(f"{CACHE} not found — run robson_analysis.py first.")

    print(f"Loading {CACHE} ...")
    df = pd.read_parquet(CACHE)

    d = df[(df["Classification_status"] == "Classified") &
           (df["Robson_group"] == "1")].copy()
    print(f"Robson group 1: {len(d):,} births\n")

    d = build_strata(d)
    d["apgar5_lt7"] = pd.to_numeric(d["APGAR5"], errors="coerce").lt(7)
    d["nicu"] = d["AB_NICU"].astype(str).str.strip() == "Y"

    d = d[d["payer"].isin(["Private", "Medicaid", "Self-pay"])]

    # ---- 1. Is the caesarean difference real, or is it case mix? -----------
    print("=" * 76)
    print("1. CAESAREAN RATE BY PAYER, WITHIN ROBSON GROUP 1")
    print("   Standardised to the group 1 risk profile")
    print("   (maternal age x BMI x diabetes x hypertensive disorder)")
    print("=" * 76)
    cs = direct_standardise(d, "payer", "is_cesarean")
    print(cs.round(2).to_string())
    print("""
  Crude vs standardised is the whole point. If the gap between payers
  survives standardisation, it is not explained by the women — it is
  explained by the practice.
""")

    # ---- 2. Do the extra caesareans buy better outcomes? -------------------
    print("=" * 76)
    print("2. NEONATAL OUTCOMES BY PAYER, SAME STANDARDISATION")
    print("=" * 76)
    for outcome, label in [("apgar5_lt7", "5-minute Apgar below 7"),
                           ("nicu", "NICU admission"),
                           ("near_miss_flag", "Maternal near-miss")]:
        print(f"\n  {label}:")
        print(direct_standardise(d, "payer", outcome).round(3).to_string())

    print("""
  How to read this. If the payer stratum with the HIGHEST standardised
  caesarean rate does NOT show correspondingly better neonatal outcomes,
  the additional caesareans are not producing a safety benefit in a
  population where every woman entered labour spontaneously at term with
  a single cephalic fetus and no uterine scar.

  That is the strongest form this argument can take on observational
  data, and it is made entirely on US records — before Egypt is
  mentioned at all.
""")

    # ---- 3. Time of day, same treatment -----------------------------------
    print("=" * 76)
    print("3. CAESAREAN RATE BY TIME OF DAY, WITHIN GROUP 1 (standardised)")
    print("=" * 76)
    print(direct_standardise(d, "time_of_day", "is_cesarean").round(2).to_string())
    print("""
  A woman's physiology does not know the hour. Any surviving gradient is
  a property of the service, not of the labour.
""")

    # ---- 4. What the standardisation actually did -------------------------
    print("=" * 76)
    print("4. AUDIT — case mix by payer (why standardisation was needed)")
    print("=" * 76)
    for v in ["age_band", "bmi_band", "diabetes", "hypertension"]:
        t = pd.crosstab(d["payer"], d[v], normalize="index") * 100
        print(f"\n  {v} (% within payer):")
        print(t.round(1).to_string())

    print("\n" + "=" * 76)
    print("STAGE 3 COMPLETE.")
    print("=" * 76)

### 2.2 — `stage4_setting_and_wider_adjustment.py`: Expanding the Confounders

**What Stage 3 left open:**

1. **Self-pay stood apart** — lowest CS rate (9.04%) but highest near-miss rate (0.630%). That pattern suggests out-of-hospital births. `BFACIL` (birth place) settles it.

2. **More confounders needed** — smoking, prenatal care timing, and education all correlate with both payer and CS risk.

**New fields added** (verified against UserGuide2023.pdf):
- `BFACIL` (birth place), `CIG_REC` (smoking), `PRECARE5` (prenatal care), `MEDUC` (education), `DMAR` (marital status), `RF_PPTERM` (previous preterm)

**Documented limitation from NCHS:** The User Guide states that births attended by certified nurse midwives are understated — some hospitals require a physician to be recorded as attendant even when no physician was present. This is NCHS confirming the artefact diagnosed in Stage 2.

**Result:** After restricting to hospital births and adjusting for 5 variables, the payer gradient flattened further. The extra CS in private insurance did not produce better Apgar scores.

In [ ]:
"""
=============================================================================
STAGE 4 — EXPANDED CONFOUNDING CONTROL
=============================================================================

WHAT STAGE 3 SHOWED, AND WHY WE ARE NOT DONE

Stage 3 standardised on maternal age, BMI, diabetes and hypertensive
disorder. The crude payer gradient (private 14.25% > Medicaid 11.97% >
self-pay 8.30%) did not survive: after standardisation Medicaid came out
HIGHEST (13.70% vs private 13.07%), because private insurance serves a
much older population and maternal age drives caesarean.

Two questions remain open, and both are answerable:

  1. Self-pay stood apart — lowest caesarean rate (9.04% standardised) but
     the HIGHEST maternal near-miss rate (0.630% vs 0.478% private). That
     pattern is what you would expect if a meaningful share of self-pay
     births are out-of-hospital. BFACIL (birth place) settles it.

  2. Age, BMI and comorbidity are not the only things that differ between
     payers. Smoking, timing of first prenatal visit and maternal education
     are all associated with both payer and caesarean risk. If the flat
     Apgar result survives adjustment for those too, it is a real finding.

NEW FIELDS ADDED (positions verified against UserGuide2023.pdf)

  BFACIL    pos 32       Birth place — 1 hospital, 2 birth centre,
                         3/4/5 home, 6 clinic, 7 other, 9 unknown
  CIG_REC   pos 269      Cigarette recode — Y/N/U
  PRECARE5  pos 227      Month prenatal care began, recoded —
                         1 = 1st trimester ... 4 = none, 5 = unknown
  MEDUC     pos 124      Mother's education, 1-8 (9 unknown)
  DMAR      pos 120      Marital status — 1 married, 2 unmarried
  RF_PPTERM pos 318      Previous preterm birth — Y/N/U

A DOCUMENTED LIMITATION, FROM NCHS ITSELF

The 2023 User Guide states that births attended by certified nurse
midwives are understated, because when more than one provider is present
the attendant is hard to identify — and that some hospitals require a
physician to be recorded as attendant even when no physician was
physically present at a midwife-attended birth.

This is the source agency confirming the artefact diagnosed in stage 2.
The attendant field is therefore reported descriptively only and is never
used as a practice-pattern comparator. Cite the User Guide for this, not
our own inference.

ALSO CONFIRMED BY THE USER GUIDE

NCHS defines its published low-risk caesarean rate as: singleton, term
(37+ weeks by obstetric estimate), cephalic caesareans to women having a
first birth, per 100 such births. That is exactly Robson groups 1 + 2a +
2b. Our 26.56% is therefore a direct comparison with a published official
statistic, not an approximation.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).parent))
from stage3_confounding import direct_standardise

CACHE = "natality_2023_filtered.parquet"
CACHE_EXT = "natality_2023_extended.parquet"

NEW_COLSPECS = [
    (31, 32),     # BFACIL      pos 32      Birth Place
    (119, 120),   # DMAR        pos 120     Marital Status
    (123, 124),   # MEDUC       pos 124     Mother's Education
    (226, 227),   # PRECARE5    pos 227     Month Prenatal Care Began Recode
    (268, 269),   # CIG_REC     pos 269     Cigarette Recode
    (317, 318),   # RF_PPTERM   pos 318     Previous Preterm Birth
]
NEW_NAMES = ["BFACIL", "DMAR", "MEDUC", "PRECARE5", "CIG_REC", "RF_PPTERM"]


# =============================================================================
def extend_cache(raw_path):
    """
    Read ONLY the new columns from the raw file and attach them to the
    existing cache by row position.

    Row alignment is safe because both reads use the same file in the same
    order with no row filtering. The row-count assertion below is the guard:
    if it ever fails, do not proceed.
    """
    if Path(CACHE_EXT).exists():
        print(f"Loading {CACHE_EXT} ...")
        return pd.read_parquet(CACHE_EXT)

    if not Path(CACHE).exists():
        raise SystemExit(f"{CACHE} not found — run robson_analysis.py first.")

    print(f"Loading {CACHE} ...")
    base = pd.read_parquet(CACHE)
    print(f"  {len(base):,} rows")

    print(f"Reading {len(NEW_NAMES)} new columns from {raw_path} ...")
    frames = []
    for chunk in pd.read_fwf(raw_path, colspecs=NEW_COLSPECS, names=NEW_NAMES,
                             dtype=str, chunksize=500_000):
        frames.append(chunk)
    new = pd.concat(frames, ignore_index=True)
    print(f"  {len(new):,} rows")

    assert len(new) == len(base), (
        f"ROW COUNT MISMATCH: cache has {len(base):,}, new read has {len(new):,}. "
        "Do not proceed — the row alignment assumption is broken."
    )

    out = pd.concat([base.reset_index(drop=True), new.reset_index(drop=True)], axis=1)
    print(f"Saving {CACHE_EXT} ...")
    out.to_parquet(CACHE_EXT, index=False)
    print("Done.\n")
    return out


def build_strata_v2(d):
    """
    Wider adjustment set than stage 3, but deliberately coarser bands so the
    number of strata stays manageable and cell coverage stays high.
    A perfectly adjusted comparison with 3% coverage is worth nothing.
    """
    age = pd.to_numeric(d["MAGER"], errors="coerce")
    d["age_band"] = pd.cut(age, [0, 19, 24, 29, 34, 99],
                           labels=["<20", "20-24", "25-29", "30-34", "35+"])

    bmi = pd.to_numeric(d["BMI"], errors="coerce")
    bmi = bmi.where(bmi < 99)
    d["bmi_band"] = pd.cut(bmi, [0, 18.5, 25, 30, 99],
                           labels=["<18.5", "18.5-24", "25-29", "30+"])

    yes = {"Y"}
    diab = (d["RF_PDIAB"].astype(str).str.strip().isin(yes) |
            d["RF_GDIAB"].astype(str).str.strip().isin(yes))
    hyp = (d["RF_PHYPE"].astype(str).str.strip().isin(yes) |
           d["RF_GHYPE"].astype(str).str.strip().isin(yes) |
           d["RF_EHYPE"].astype(str).str.strip().isin(yes))
    d["comorbid"] = (diab | hyp)

    d["smoker"] = d["CIG_REC"].astype(str).str.strip() == "Y"

    pc = d["PRECARE5"].astype(str).str.strip()
    d["prenatal"] = np.select(
        [pc == "1", pc.isin(["2", "3"]), pc == "4"],
        ["1st trimester", "later", "none"],
        default="unknown")

    d["stratum"] = (d["age_band"].astype(str) + "|" + d["bmi_band"].astype(str)
                    + "|" + d["comorbid"].astype(str) + "|" + d["smoker"].astype(str)
                    + "|" + d["prenatal"])
    return d


# =============================================================================
if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("path", nargs="?", default="Nat2023PublicUS.txt")
    args = p.parse_args()

    df = extend_cache(args.path)

    d = df[(df["Classification_status"] == "Classified") &
           (df["Robson_group"] == "1")].copy()
    d["apgar5_lt7"] = pd.to_numeric(d["APGAR5"], errors="coerce").lt(7)
    d["nicu"] = d["AB_NICU"].astype(str).str.strip() == "Y"

    bf = d["BFACIL"].astype(str).str.strip()
    d["setting"] = np.select(
        [bf == "1", bf == "2", bf.isin(["3", "4", "5"]), bf.isin(["6", "7"])],
        ["Hospital", "Birth centre", "Home", "Clinic/other"],
        default="Unknown")

    # ---- 1. Does birth place explain self-pay? ----------------------------
    print("=" * 78)
    print("1. WHERE DO THESE BIRTHS HAPPEN? (Robson group 1, by payer)")
    print("=" * 78)
    print(f"\nGroup 1 total: {len(d):,}\n")
    print("  Setting distribution overall:")
    print((100 * d["setting"].value_counts(normalize=True)).round(3).to_string())
    print("\n  Setting by payer (% within payer):")
    print((100 * pd.crosstab(d["payer"], d["setting"], normalize="index")).round(2).to_string())
    print("\n  Caesarean rate and near-miss by setting:")
    t = d.groupby("setting").agg(n=("is_cesarean", "size"),
                                 cs_rate=("is_cesarean", "mean"),
                                 near_miss=("near_miss_flag", "mean"),
                                 apgar5_lt7=("apgar5_lt7", "mean"))
    for c in ["cs_rate", "near_miss", "apgar5_lt7"]:
        t[c] = (100 * t[c]).round(3)
    print(t.to_string())
    print("""
  If self-pay is disproportionately out-of-hospital, its low caesarean
  rate and high near-miss rate are a property of the SETTING, not of the
  payment method — and the comparison must be re-run inside hospitals only.
""")

    # ---- 2. Hospital-only, wider adjustment -------------------------------
    h = d[d["setting"] == "Hospital"].copy()
    h = build_strata_v2(h)
    h = h[h["payer"].isin(["Private", "Medicaid", "Self-pay"])]

    print("=" * 78)
    print("2. HOSPITAL BIRTHS ONLY — WIDER ADJUSTMENT")
    print("   age x BMI x comorbidity x smoking x prenatal-care timing")
    print("=" * 78)
    print(f"\n  Hospital group 1 births: {len(h):,}\n")
    print("  Caesarean rate:")
    print(direct_standardise(h, "payer", "is_cesarean").round(2).to_string())

    for outcome, label in [("apgar5_lt7", "5-minute Apgar below 7"),
                           ("nicu", "NICU admission"),
                           ("near_miss_flag", "Maternal near-miss")]:
        print(f"\n  {label}:")
        print(direct_standardise(h, "payer", outcome).round(3).to_string())

    print("""
  THE TEST. Compare the caesarean row against the Apgar row.

  If the payer groups differ materially in standardised caesarean rate but
  not in standardised 5-minute Apgar below 7, then within hospitals, among
  nulliparous women at term with a single cephalic fetus who entered labour
  spontaneously and have no uterine scar, and holding age, BMI, comorbidity,
  smoking and prenatal care constant — the extra caesareans are not
  producing a measurable neonatal benefit.

  Caveat to keep: NICU admission is not a neutral outcome here. Caesarean
  delivery itself causes transient tachypnoea of the newborn and raises NICU
  admission. NICU is partly a CONSEQUENCE of the intervention, not an
  independent measure of whether it was needed. Apgar is the cleaner
  endpoint of the two.
""")

    # ---- 3. Time of day, hospital only ------------------------------------
    print("=" * 78)
    print("3. TIME OF DAY — HOSPITAL BIRTHS ONLY, WIDER ADJUSTMENT")
    print("=" * 78)
    print(direct_standardise(h, "time_of_day", "is_cesarean").round(2).to_string())
    print("""
  Clinical caveat, stated before anyone else raises it: spontaneous labour
  has a circadian rhythm, with onset and delivery clustering overnight. A
  lower caesarean rate in the small hours may be physiology rather than
  decision-making. The evening rise is the harder half to explain that way,
  but this analysis cannot separate fatigue, shift handover and staffing
  from chance. Report the gradient; do not name a cause.
""")

    # ---- 4. Audit -------------------------------------------------------
    print("=" * 78)
    print("4. AUDIT — what differs between payers, hospital births only")
    print("=" * 78)
    for v in ["age_band", "smoker", "prenatal", "comorbid"]:
        print(f"\n  {v} (% within payer):")
        print((100 * pd.crosstab(h["payer"], h[v], normalize="index")).round(1).to_string())

    print("\n" + "=" * 78)
    print("STAGE 4 COMPLETE.")
    print("=" * 78)

### 2.3 — `egypt_standardisation.py`: The Same Test on Egyptian Data

**The key comparison:**
- **US (Group 1):** Crude private-public gap disappeared after standardisation
- **Egypt (DHS facility births):** Crude 21.5pp gap. After standardising on wealth × age × parity → gap shrinks but **survives**

**Two fixes over the first version:**
1. Reads `EGIR61FL.DTA` directly (the intermediate CSV had duplicated columns)
2. Maternal age computed EXACTLY at date of each birth from century-month codes — not age at interview, which would leak 4 years of error into the standardisation

**Home births excluded:** 13.2% of Egyptian births with 0% CS rate by definition (a home birth that becomes a CS is recorded as a facility birth).

In [ ]:
"""
=============================================================================
EGYPT — THE SAME TEST, RUN ON EGYPTIAN DATA  (v2)
=============================================================================
    python egypt_standardisation.py

TWO FIXES OVER THE FIRST VERSION

1. Reads EGIR61FL.DTA directly instead of the intermediate CSV, which had
   duplicated column names.

2. Maternal age is now computed EXACTLY at the date of each birth, from the
   century-month codes: (child's birth CMC - mother's birth CMC) / 12.
   The first version fell back to age at interview, which is wrong here and
   not harmlessly wrong: a woman interviewed in 2014 about a birth in 2010
   is recorded four years too old, and since age drives caesarean risk that
   error would have leaked straight into the standardisation it is meant to
   remove.

WHAT THIS TESTS
On US data the crude caesarean gap between payers inside Robson group 1 did
not survive adjustment — private 14.25% and self-pay 8.30% became 13.26%
and 13.56%. EDHS 2014 shows a crude private-versus-public gap of 21.5
points. Egyptian private-sector mothers are wealthier, more urban and lower
parity, all of which raise caesarean risk on their own, so the crude figure
means nothing yet.

direct_standardise is IMPORTED from stage3_confounding.py, not rewritten.
The two countries are analysed by the same function, not merely by similar
methods.

Home births are excluded: 13.2% of Egyptian births with a 0.00% caesarean
rate by definition, since a home birth that becomes a caesarean is recorded
as a facility birth. This mirrors the hospital-only restriction used on the
US data.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).parent))
try:
    from stage3_confounding import direct_standardise
    print("direct_standardise imported from stage3_confounding.py")
    print("  — the identical function applied to the CDC data.\n")
except Exception as e:
    raise SystemExit(f"Could not import stage3_confounding.py: {e}")

DTA = next((p for p in Path(".").glob("*.DTA") if "IR61" in p.name.upper()), None)
if DTA is None:
    raise SystemExit("EGIR61FL.DTA not found in this folder.")

WOMAN = ["caseid", "v005", "v011", "v024", "v025", "v106", "v190"]
BIRTH = ["m17", "m15", "bord", "b0", "b3"]
want = (WOMAN + [f"{x}_{i}" for x in BIRTH for i in range(1, 7)]
              + [f"{x}_{i:02d}" for x in BIRTH for i in range(1, 7)])

print(f"Loading {DTA.name} ...")
head = pd.read_stata(DTA, convert_categoricals=False, chunksize=1)
available = set(next(head).columns)
use = [c for c in want if c in available]
raw = pd.read_stata(DTA, columns=use, convert_categoricals=False)
print(f"  {len(raw):,} women, {len(use)} variables kept\n")

# ---- reshape to one row per birth ----------------------------------------
frames = []
for i in range(1, 7):
    ren = {}
    for x in BIRTH:
        for cand in (f"{x}_{i}", f"{x}_{i:02d}"):
            if cand in raw.columns:
                ren[cand] = x
                break
    if "m17" not in ren.values():
        continue
    keep = [c for c in WOMAN if c in raw.columns]
    sub = raw[keep + list(ren)].rename(columns=ren).copy()
    sub["birth_index"] = i
    frames.append(sub)

b = pd.concat(frames, ignore_index=True)
b = b.loc[:, ~b.columns.duplicated()]
b = b[b["m17"].notna()].copy()

b["weight"] = pd.to_numeric(b["v005"], errors="coerce") / 1_000_000
b["cs"] = (pd.to_numeric(b["m17"], errors="coerce") == 1).astype(float)
print(f"Births with a recorded mode of delivery: {len(b):,}")
print(f"Weighted caesarean rate (all births)   : "
      f"{100*(b['cs']*b['weight']).sum()/b['weight'].sum():.2f}%   (published 51.8%)\n")

# ---- sector ---------------------------------------------------------------
m15 = pd.to_numeric(b["m15"], errors="coerce")
b["sector"] = np.select(
    [m15.between(11, 12), m15.between(21, 26), m15.between(31, 36)],
    ["Home", "Public", "Private"], default="Other")

# ---- EXACT maternal age at the birth --------------------------------------
b3   = pd.to_numeric(b["b3"],   errors="coerce")
v011 = pd.to_numeric(b["v011"], errors="coerce")
b["age_at_birth"] = (b3 - v011) / 12.0
ok = b["age_at_birth"].between(10, 55)
print(f"Maternal age at birth computed for {100*ok.mean():.1f}% of births "
      f"(median {b.loc[ok,'age_at_birth'].median():.1f} years)\n")
b.loc[~ok, "age_at_birth"] = np.nan
b["age_band"] = pd.cut(b["age_at_birth"], [0, 19, 24, 29, 34, 99],
                       labels=["<20", "20-24", "25-29", "30-34", "35+"])

# ---- parity, wealth, residence -------------------------------------------
b["parity_band"] = pd.cut(pd.to_numeric(b["bord"], errors="coerce"),
                          [0, 1, 3, 99], labels=["1st", "2nd-3rd", "4th+"])
b["wealth"] = pd.to_numeric(b["v190"], errors="coerce")
b["urban"]  = pd.to_numeric(b["v025"], errors="coerce")

b["stratum"] = (b["wealth"].astype(str) + "|" + b["age_band"].astype(str)
                + "|" + b["parity_band"].astype(str))

f = b[b["sector"].isin(["Public", "Private"])].copy()
print(f"Facility births analysed: {len(f):,} "
      f"({100*len(f)/len(b):.1f}% of births; home births excluded)\n")

# =============================================================================
print("=" * 74)
print("1. CAESAREAN RATE BY SECTOR — CRUDE vs STANDARDISED")
print("   Strata: wealth quintile x maternal age at birth x parity")
print("=" * 74)
r = direct_standardise(f, "sector", "cs", min_cell=25)
print(r.round(2).to_string())

if {"Private", "Public"} <= set(r.index):
    crude = r.loc["Private","crude_%"] - r.loc["Public","crude_%"]
    std   = r.loc["Private","standardised_%"] - r.loc["Public","standardised_%"]
    print(f"\n  Private minus public, crude        : {crude:+.2f} pp")
    print(f"  Private minus public, standardised : {std:+.2f} pp")
    if crude:
        print(f"  Explained by case mix              : {crude-std:+.2f} pp "
              f"({100*(crude-std)/crude:.0f}% of the crude gap)")

print("\n" + "-" * 74)
print("Sensitivity — urban/rural added to the strata")
print("-" * 74)
f2 = f.copy(); f2["stratum"] = f2["stratum"] + "|" + f2["urban"].astype(str)
print(direct_standardise(f2, "sector", "cs", min_cell=20).round(2).to_string())

print("\n" + "-" * 74)
print("Sensitivity — first births only (the cleanest comparison available)")
print("-" * 74)
f3 = f[f["parity_band"] == "1st"].copy()
f3["stratum"] = f3["wealth"].astype(str) + "|" + f3["age_band"].astype(str)
print(f"  n = {len(f3):,}")
print(direct_standardise(f3, "sector", "cs", min_cell=20).round(2).to_string())
print("""
  First births carry no previous caesarean, so this removes the group 5
  effect entirely and isolates how each sector treats a woman with no
  uterine scar. It is the closest Egyptian equivalent to Robson group 1
  that DHS can support.
""")

# =============================================================================
print("=" * 74)
print("2. AUDIT — what differs between the sectors")
print("=" * 74)
for v, name in [("wealth","wealth quintile"), ("age_band","maternal age at birth"),
                ("parity_band","parity"), ("urban","residence (1 urban, 2 rural)")]:
    print(f"\n  {name} (% within sector):")
    print((100*pd.crosstab(f["sector"], f[v], normalize="index")).round(1).to_string())

print("\n" + "=" * 74)
print("3. THE COMPARISON THE PROJECT RESTS ON")
print("=" * 74)
print("""
  United States, Robson group 1, by payer
    crude          private 14.25%   Medicaid 11.97%   self-pay  8.30%
    standardised   private 13.26%   Medicaid 13.90%   self-pay 13.56%
    -> the gap disappears

  Egypt, facility births, by sector — see section 1 above.
""")

f.to_csv("egypt_sector_analysis.csv", index=False)
print("Saved egypt_sector_analysis.csv")

---

## Part 3 — `egypt_gap_v2.py`: The Main Result

**This is the most important script in the entire project.** It produces the corrected indirect standardisation that quantifies Egypt's caesarean excess.

### The three corrections (all three reduced the estimate, not raised it)

**CHANGE 1 — Eight categories, not ten**

Robson groups 1 vs 2 and 3 vs 4 are separated by onset of labour — a clinical DECISION, not a patient characteristic. A woman does not arrive as "Group 2". She arrives nulliparous with a single cephalic fetus at term, and then the hospital chooses whether to induce, allow spontaneous labour, or perform a pre-labour CS.

Standardising on Egypt's Group 2 share therefore holds Egyptian PRACTICE constant while calling it case-mix. Egypt barely induces — Qena recorded **three inductions in 2,186 births**, against 34.5% in the US.

Collapsing the onset splits gives 8 categories: (1+2), (3+4), 5, 6, 7, 8, 9, 10.

**Useful consequence:** The reference rate for (1+2) = 26.57%, which is the NTSV rate NCHS publishes. The yardstick is a published statistic, not something derived here.

**CHANGE 2 — Group 5 reallocated to Groups 3+4 only**

Scenario B asks: what if Egypt's Group 5 share matched the reference? The old code spread freed women proportionally across ALL other groups — putting some into Groups 6, 7, 8, 10 (breech, twins, preterm, with rates 43-96%).

**That is impossible.** A woman in Group 5 is multiparous, single, cephalic, term. Remove the scar → she is still multiparous, single, cephalic, term = Groups 3+4. A scar does not change fetal presentation, plurality, or gestational age.

On Benha data, the old method misplaced 6.6% of all births into impossible groups and understated the gap by 5.2pp.

**CHANGE 3 — Group 9 reference rate set to 100%**

Transverse/oblique lie cannot deliver vaginally. The CDC-derived rate (46.52%) was contaminated by the "other presentation" catch-all. Corrected to the definitional value. Effect: ~0.2pp, running against the project's conclusion.

### What was considered and rejected

A Group 10 scenario (resetting preterm share). Rejected because unlike Group 5, preterm births have no single destination — Group 10 contains nulliparous women, multiparous women, and women with previous CS. And most preterm birth is spontaneous, not iatrogenic. Reported as a diagnostic observation, not a scenario.

In [ ]:
"""
=============================================================================
EGYPT GAP — indirect standardisation on eight case-mix categories
=============================================================================
    python egypt_gap_v2.py

Replaces egypt_gap.py. Three changes came out of a methodological review;
all three made the calculation simpler, and all three moved the estimate
DOWN, so the figures reported here are more conservative than the ones they
replace.

CHANGE 1 — EIGHT CATEGORIES, NOT TEN

Robson is not a pure case-mix stratification. Groups 6 to 10 are properties
of the woman — breech, twins, transverse lie, preterm. But the split of 1
from 2, and of 3 from 4, is a clinical DECISION: was she allowed to labour,
induced, or sectioned before labour began?

A woman does not arrive as "group 2". She arrives nulliparous with a single
cephalic fetus at term, and then the hospital chooses.

Standardising on Egypt's group 2 share therefore holds Egyptian PRACTICE
constant while calling it case mix. Egypt barely induces — Qena recorded
three inductions in 2,186 births, against 34.5% of all US births. No
difference in the women explains that.

So the two onset splits are collapsed and the standardisation runs on eight
categories: (1+2), (3+4), 5, 6, 7, 8, 9, 10. The ten-group table is still
reported, for diagnosis; it is simply not what the calculation runs on.

A useful consequence: the reference rate for (1+2) is 26.57%, which is the
NTSV rate NCHS publishes independently. The yardstick is a published
statistic rather than something derived here.

CHANGE 2 — GROUP 5 IS REALLOCATED TO GROUPS 3+4, NOT SPREAD PROPORTIONALLY

Scenario B asks what Egypt's expected rate would be if its group 5 share
matched the reference. The old code answered by rescaling the other nine
groups proportionally, which put freed women into groups 6, 7, 8 and 10 —
breech, twins, preterm, with rates from 43% to 96%.

That is not possible. A woman in group 5 is multiparous, single, cephalic
and at term. Remove the uterine scar and she is still multiparous, single,
cephalic and at term — which is the definition of groups 3 and 4. A scar
does not change fetal presentation, plurality or gestational age.

This is derivation from the definitions, not estimation. On the Benha data
the old method misplaced 6.6% of all births into groups that cannot contain
them, and it understated the gap by 5.2 percentage points.

CHANGE 3 — GROUP 9 REFERENCE RATE SET TO 100%

WHO group 9 is transverse or oblique lie, which cannot deliver vaginally.
Our CDC-derived rate of 46.52% is contaminated: CDC folds face, brow and
compound presentations into the same "other" code. That limitation was
documented early and then, inconsistently, the contaminated rate was still
used in the standardisation. It is corrected here to the definitional
value. The effect is about 0.2 percentage points, and it runs against the
project's own conclusion.

WHAT WAS CONSIDERED AND REJECTED

A third scenario resetting group 10 to the reference share. Group 10 in
Qena is 22.14% against a reference 7.46%, and a planned caesarean at 37
weeks does manufacture a preterm birth — Egypt's own ministry warned
against exactly that in April 2026.

But group 5 and group 10 are not analogous. Every woman in group 5 is there
because of a previous caesarean; there is no such thing as a spontaneous
group 5. Most preterm birth, by contrast, is spontaneous, and the share
that is iatrogenic cannot be identified in this data. Reallocating group 10
also has no clean destination: it contains nulliparous women, multiparous
women and women with previous caesareans, so freed women would belong in
1+2, 3+4 or 5 depending on facts we do not have.

Group 10 is therefore reported as a diagnostic observation and a stated
limitation, and no scenario is built on it.
=============================================================================
"""

import pandas as pd
import numpy as np
import json

# =============================================================================
# REFERENCE — CDC 2023, ten groups
# =============================================================================
REF_SIZE = {"1":16.43,"2":17.49,"3":21.98,"4":16.66,"5":12.53,
            "6":1.72,"7":1.73,"8":3.16,"9":0.83,"10":7.46}
REF_RATE = {"1":13.00,"2":39.31,"3":2.52,"4":18.46,"5":83.38,
            "6":96.24,"7":93.78,"8":75.07,"9":100.00,"10":43.17}
#                                                     ^ definitional, see above

def collapse(a, b):
    return (REF_SIZE[a]*REF_RATE[a] + REF_SIZE[b]*REF_RATE[b]) / (REF_SIZE[a]+REF_SIZE[b])

R8 = {"1+2": collapse("1","2"), "3+4": collapse("3","4"),
      **{g: REF_RATE[g] for g in ["5","6","7","8","9","10"]}}
R8_SIZE = {"1+2": REF_SIZE["1"]+REF_SIZE["2"], "3+4": REF_SIZE["3"]+REF_SIZE["4"],
           **{g: REF_SIZE[g] for g in ["5","6","7","8","9","10"]}}
CATS = ["1+2","3+4","5","6","7","8","9","10"]

# =============================================================================
# EGYPTIAN SOURCES — ten-group sizes as published
# =============================================================================
EGYPT = {
    "Benha 2018": {
        "n": 850, "observed": 54.71,
        "size": {"1":13.18,"2":3.29,"3":26.71,"4":2.71,"5":36.24,
                 "6":4.94,"7":3.29,"8":2.35,"9":0.24,"10":7.06},
        "note": "prospective, WHO manual, all births >=28 weeks including stillbirths",
    },
    "Qena 2022-23": {
        "n": 2186, "observed": 79.69,
        "size": {"1":3.75,"2":7.41,"3":8.78,"4":3.75,"5":40.81,
                 "6":0.91,"7":5.35,"8":6.68,"9":0.41,"10":22.14},
        "note": "cross-sectional; cite Table 3 only, N=2,186 not the 2,000 stated",
    },
}

def to8(m):
    return {"1+2": m["1"]+m["2"], "3+4": m["3"]+m["4"],
            **{g: m[g] for g in ["5","6","7","8","9","10"]}}

def expected(mix8, rates=R8):
    return sum(mix8[g]/100 * rates[g] for g in mix8)

def reset_group5(mix8, target):
    """Freed women go to 3+4 and nowhere else. See CHANGE 2 above."""
    out = dict(mix8)
    freed = mix8["5"] - target
    out["5"] = target
    out["3+4"] = mix8["3+4"] + freed
    return out

# =============================================================================
print("=" * 76)
print("REFERENCE — eight case-mix categories, CDC 2023")
print("=" * 76)
ref = pd.DataFrame({"category": CATS,
                    "reference_size_pct": [round(R8_SIZE[g],2) for g in CATS],
                    "reference_cs_rate_pct": [round(R8[g],2) for g in CATS]})
print(ref.to_string(index=False))
print(f"""
  The (1+2) rate of {R8['1+2']:.2f}% is the NTSV measure NCHS publishes. This
  project reproduced it independently at 26.56%, so the yardstick is a
  published national statistic rather than a figure derived here.
""")

print("=" * 76)
print("MAIN RESULT")
print("=" * 76)
rows = []
for name, d in EGYPT.items():
    m8 = to8(d["size"])
    assert abs(sum(m8.values()) - 100) < 0.15, f"{name} shares do not sum to 100"
    a = expected(m8)
    b8 = reset_group5(m8, R8_SIZE["5"])
    assert abs(sum(b8.values()) - 100) < 0.15
    b = expected(b8)
    obs = d["observed"]
    rows.append({"source": name, "n": d["n"], "observed_pct": obs,
                 "expected_A_pct": round(a,2), "gap_A_pp": round(obs-a,2),
                 "expected_B_pct": round(b,2), "gap_B_pp": round(obs-b,2),
                 "inherited_pp": round(a-b,2),
                 "inherited_share_of_total_pct": round(100*(a-b)/(obs-b))})
main = pd.DataFrame(rows)
print(main.to_string(index=False))
print("""
  A            Egypt's case mix as observed — excess attributable to current
               practice
  B            group 5 share set to the reference — total accumulated excess
  inherited    the difference: excess carried by a case mix that earlier
               caesareans created
""")

print("=" * 76)
print("THE DIAGNOSTIC FINDING")
print("=" * 76)
for r in rows:
    print(f"  {r['source']:<14} inherited {r['inherited_share_of_total_pct']:>3}% of the total excess")
print("""
  The two hospitals differ less in HOW MUCH excess they carry than in where
  it comes from. At Benha current practice sits close to the reference and
  almost all the excess is inherited; at Qena it is roughly evenly split.

  That points to different interventions. Benha needs to stop new group 5
  forming — which means the caesarean rate in first births. Qena needs that
  and a change in current practice as well.
""")

# =============================================================================
print("=" * 76)
print("SENSITIVITY — the group 5 share, read as a policy trajectory")
print("=" * 76)
print("""  Not a statement of uncertainty. We know Egypt's group 5 share. This
  answers a different question: if it fell to X, what would follow?
""")
print(f"{'group 5 share':>15}", end="")
for n in EGYPT: print(f"{n:>18}", end="")
print()
print("-" * 52)
sens = []
for g5 in [12.53, 18, 24, 30, 36, 40.81]:
    print(f"{g5:>14.2f}%", end="")
    for name, d in EGYPT.items():
        m8 = to8(d["size"])
        if g5 > m8["5"]:
            print(f"{'—':>18}", end=""); continue
        e = expected(reset_group5(m8, g5))
        gap = d["observed"] - e
        sens.append({"group5_share_pct": g5, "source": name,
                     "expected_pct": round(e,2), "gap_pp": round(gap,2)})
        print(f"{gap:>17.1f}", end="")
    print()

print("""
  Every step toward the reference share widens the gap, because a previous
  caesarean is the strongest single predictor in the table and removing it
  moves a woman from an 83% category to a 9% one.
""")

# =============================================================================
print("=" * 76)
print("SENSITIVITY — what if the reference itself is too permissive")
print("=" * 76)
print("""  The United States runs at 32%, above any clinical ideal. Scaling the
  reference rates down represents a lower-intervention standard.
""")
print(f"{'reference':>18}", end="")
for n in EGYPT: print(f"{n:>18}", end="")
print()
print("-" * 54)
scale_rows = []
for f, lab in [(1.00, "as observed"), (0.85, "-15%"), (0.70, "-30%")]:
    scaled = {g: v*f for g, v in R8.items()}
    print(f"{lab:>18}", end="")
    for name, d in EGYPT.items():
        e = expected(to8(d["size"]), scaled)
        gap = d["observed"] - e
        scale_rows.append({"scaling": lab, "source": name, "gap_pp": round(gap,2)})
        print(f"{gap:>17.1f}", end="")
    print()

allgaps = ([r["gap_A_pp"] for r in rows] + [r["gap_B_pp"] for r in rows]
           + [s["gap_pp"] for s in sens] + [s["gap_pp"] for s in scale_rows])
print(f"""
=============================================================================
THE INTERVAL
=============================================================================
  Across every combination tested, the estimated excess runs from
  {min(allgaps):.1f} to {max(allgaps):.1f} percentage points, and it never closes.

  Every bias that could not be tested runs the same way. A nationally
  representative case mix would carry a smaller group 5 and widen the gap.
  A lower-intervention reference would widen it. Referral bias in two
  tertiary hospitals narrows it. And all three methodological corrections
  applied in this version reduced the estimate rather than raising it.
""")

print("=" * 76)
print("REPORTED SEPARATELY, NOT AS A SCENARIO")
print("=" * 76)
for name, d in EGYPT.items():
    print(f"  {name:<14} group 10 share {d['size']['10']:>6.2f}%   "
          f"reference {REF_SIZE['10']:.2f}%")
print("""
  Qena's preterm share is three times the reference. A planned caesarean at
  37 weeks creates a preterm birth, and Egypt's ministry warned against
  precisely that in April 2026, so part of this share is likely iatrogenic.

  No scenario is built on it. Most preterm birth is spontaneous, the
  iatrogenic share cannot be identified here, and freed women would have no
  single destination — group 10 contains nulliparous women, multiparous
  women and women with previous caesareans alike.
""")

main.to_csv("egypt_gap_main.csv", index=False)
pd.DataFrame(sens).to_csv("egypt_gap_sensitivity_group5.csv", index=False)
ref.to_csv("egypt_reference_8cat.csv", index=False)

mix_rows = []
for name, d in EGYPT.items():
    m8 = to8(d["size"])
    for g in CATS:
        mix_rows.append({"source": name, "group_8": g,
                         "share_pct": round(m8[g], 2),
                         "reference_rate_pct": round(R8[g], 2),
                         "reference_share_pct": round(R8_SIZE[g], 2)})
pd.DataFrame(mix_rows).to_csv("egypt_case_mix_8cat.csv", index=False)

json.dump({"reference_8cat": {g: round(R8[g],4) for g in CATS},
           "reference_size_8cat": {g: round(R8_SIZE[g],4) for g in CATS},
           "main": rows, "sensitivity_group5": sens,
           "sensitivity_reference_scaling": scale_rows,
           "interval_pp": [float(min(allgaps)), float(max(allgaps))]},
          open("egypt_gap_analysis.json","w"), indent=2)

print("Saved egypt_gap_main.csv, egypt_gap_sensitivity_group5.csv,")
print("      egypt_reference_8cat.csv, egypt_case_mix_8cat.csv,")
print("      egypt_gap_analysis.json")
print("""
  egypt_case_mix_8cat.csv is the table Power BI loads alongside the births
  data. Sixteen rows — eight categories for each of two sources — carrying
  Egypt's share and the reference rate. The standardisation measure runs
  across the two tables using TREATAS, and a what-if parameter on the group
  5 share drives the trajectory shown above.
""")

### The Final Numbers

| | Observed | Expected (A) | **Gap A** | Expected (B) | **Gap B** | **Inherited** | **Inherited %** |
|---|---|---|---|---|---|---|---|
| **Benha** | 54.71% | 50.25% | **+4.46 pp** | 32.70% | +22.01 pp | 17.55 pp | **80%** |
| **Qena** | 79.69% | 59.04% | **+20.65 pp** | 38.12% | +41.57 pp | 20.92 pp | **36%** |

**The diagnostic finding:**
- **Benha: 80% of the excess is inherited** — current practice sits close to the reference. The problem is that earlier CS in first births created a massive Group 5.
- **Qena: 36% inherited** — Qena has both problems: inherited excess AND high current-practice excess.

**Different interventions needed:** Benha → stop new Group 5 forming (reduce first-birth CS). Qena → that, plus change current practice.

---

## Part 4 — Supplementary Analyses

### 4.1 — `egypt_cost.py`: The Economic Burden

**Why this was deferred and then un-deferred:** The supervisor's objection was that the proposal promised economic output while citing no cost source. The Population Council (2018) study has now been located, with sourced figures.

**The key trap:** The study reports 2014 EGP costs. Between 2014 and 2025 the pound fell from ~7 to ~48 against the dollar. Multiplying a 2014 unit cost by 2025 births would be wrong by a factor of several.

**What the model does:**
1. Reproduces the 2014 estimate from 2014 costs (defensible)
2. Shows 2025 prices descriptively (no 2025 differential possible — no vaginal-delivery comparator)

**What it deliberately does NOT claim:** A "saving." Reducing CS does not free money one-for-one (staff and facilities are fixed costs). The honest quantity is the cost differential attached to excess CS.

**A notable finding about the published figure:** The Population Council's "over EGP 900 million" cannot be built from the cost differential (EGP 412 per CS). It was built from the FULL CS cost (EGP 1,076), which overstates the burden by ~2.6x. This project reports differential-based figures and states the difference.

In [ ]:
"""
=============================================================================
EGYPT COST MODEL
=============================================================================
    python egypt_cost.py

WHY THIS WAS DEFERRED, AND WHY IT IS NOW POSSIBLE

The supervisor's objection to the economic simulator was specific: the
proposal promised output calibrated on Egyptian cost data while citing no
cost source anywhere. That objection was correct. It is now answered — the
Population Council study the supervisor named by title has been located and
its figures are below, each with a source attached.

THE FIGURE THAT MATTERS MOST, AND ITS TRAP

Population Council (2018), costing 2014 practice:
    health-system cost per caesarean ....... EGP 1,076
    health-system cost per vaginal birth ... EGP   664
    difference ............................. EGP   412
    estimated cost of caesarean over-use in 2014 ... over EGP 900 million

Those are 2014 Egyptian pounds. Between 2014 and 2025 the pound fell from
roughly 7 to roughly 48 against the dollar. Multiplying a 2014 unit cost by
a 2025 birth count would produce a number that is wrong by a factor of
several, and wrong in a direction that understates the burden.

So this model does two separate things and never mixes them:

  1. REPRODUCES the 2014 estimate from 2014 costs and 2014 rates. This is
     the defensible published figure and it is reported as such.

  2. Rescales to 2025 using 2025 PRICES from a separate source, clearly
     labelled, with the caveat that these are prices charged rather than
     costs incurred — a different quantity that includes margin.

WHAT THIS MODEL DELIBERATELY DOES NOT CLAIM

It does not claim a saving. Reducing caesarean rates does not free the money
one-for-one: staff, theatres and buildings are largely fixed in the short
run. The honest quantity is the cost DIFFERENTIAL attached to caesareans in
excess of clinical need, which is what an economist would call the resource
burden rather than a recoverable saving. That distinction is stated in the
output rather than left for a reader to notice.
=============================================================================
"""

import pandas as pd
import json

# =============================================================================
# SOURCED FIGURES
# =============================================================================
COSTS = [
    # (label, caesarean, vaginal, currency/year, perspective, source)
    ("Health-system cost", 1076, 664, "EGP 2014", "provider cost: staff time, "
     "medicines, supplies, equipment",
     "Abdel-Tawab NG, Oraby D, Hassanein N, El-Nakib S. Caesarean section "
     "deliveries in Egypt: trends, practices, perceptions and cost. "
     "Population Council, 2018 (doi 10.31899/rh6.1004)"),
    ("Private facility charge (low)", 2000, 1000, "EGP 2018", "price charged",
     "Population Council 2018, as reported in The National, Sept 2022"),
    ("Private facility charge (high)", 5000, 2000, "EGP 2018", "price charged",
     "Population Council 2018, as reported in The National, Sept 2022"),
    ("Public hospital price (low)", 3000, None, "EGP 2025", "price charged",
     "Zawia3 report, citing recent Egyptian pricing"),
    ("Public hospital price (high)", 6000, None, "EGP 2025", "price charged",
     "Zawia3 report, citing recent Egyptian pricing"),
    ("Private facility price (low)", 15000, None, "EGP 2025", "price charged",
     "Zawia3 report, citing recent Egyptian pricing"),
    ("Private facility price (high)", 30000, None, "EGP 2025", "price charged",
     "Zawia3 report, citing recent Egyptian pricing"),
]

PUBLISHED_2014 = {
    "cs_rate": 52.0,
    "annual_births": 2_600_000,
    "overuse_cost_egp": 900_000_000,
    "source": "Population Council 2018",
}

# =============================================================================
print("=" * 78)
print("SOURCED EGYPTIAN COST FIGURES")
print("=" * 78)
c = pd.DataFrame(COSTS, columns=["item","caesarean_egp","vaginal_egp",
                                 "currency_year","perspective","source"])
c["difference_egp"] = c.caesarean_egp - c.vaginal_egp
print(c[["item","caesarean_egp","vaginal_egp","difference_egp",
         "currency_year","perspective"]].to_string(index=False))
print("""
  Note the perspective column. Provider COST and price CHARGED are different
  quantities: the second includes margin and, in Egyptian private practice,
  a separate payment to the doctor. Only the first row is a cost estimate;
  the rest are prices. They are not interchangeable and are not combined.
""")

# =============================================================================
print("=" * 78)
print("1. REPRODUCING THE PUBLISHED 2014 ESTIMATE")
print("=" * 78)

delta_2014 = 1076 - 664
births = PUBLISHED_2014["annual_births"]
cs_rate = PUBLISHED_2014["cs_rate"] / 100

print(f"""
  Annual births (2014, approximate)     : {births:,}
  Caesarean rate                        : {100*cs_rate:.0f}%
  Caesareans performed                  : {births*cs_rate:,.0f}
  Cost differential per caesarean       : EGP {delta_2014:,}
""")

print(f"  {'excess share':>14} {'excess caesareans':>20} {'differential cost':>22}")
print("  " + "-"*58)
rows_2014 = []
for pct in [10, 20, 30, 40, 50]:
    n_excess = births * cs_rate * pct/100
    cost = n_excess * delta_2014
    rows_2014.append({"excess_share_pct": pct, "n_excess": int(n_excess),
                      "cost_egp_2014": int(cost)})
    print(f"  {pct:>13}% {n_excess:>20,.0f} {'EGP ' + format(cost, ',.0f'):>22}")

print("""
  THE PUBLISHED FIGURE DOES NOT RECONCILE WITH THIS ARITHMETIC, AND THAT
  IS INFORMATIVE

  Population Council report "over EGP 900 million" for 2014. Dividing that
  by the EGP 412 differential implies 2.18 million excess caesareans — more
  than the 1.35 million caesareans performed in total that year. The figure
  therefore cannot be built from the cost differential.

  Reconstructing it: valuing excess caesareans at the FULL caesarean cost of
  EGP 1,076, and defining excess as everything above a WHO reference
  threshold, reproduces it closely:

      above 19% ..... 858,000 x EGP 1,076 = EGP 923 million
      above 15% ..... 962,000 x EGP 1,076 = EGP 1,035 million

  Valuing at full cost overstates the economic burden, because those women
  would have delivered either way. Had they delivered vaginally the system
  would still have spent EGP 664 each. The quantity attributable to the
  DECISION is the difference, not the whole procedure.

  This project therefore reports differential-based figures, which are
  roughly 2.6 times smaller than the published estimate. Being more
  conservative than the source is a deliberate choice and is stated as one.
""")

# =============================================================================
print("=" * 78)
print("2. THE SAME CALCULATION ON THE PROJECT'S OWN GAP ESTIMATE")
print("=" * 78)
print("""
  The standardisation put the excess between 5.0 and 38.4 percentage points
  of the caesarean rate, depending on assumptions. Converting that to a
  count rather than a share of caesareans:
""")
print(f"  {'gap (pp)':>10} {'excess caesareans':>20} {'2014 differential cost':>26}")
print("  " + "-"*58)
rows_gap = []
for gap in [5.0, 7.6, 12.8, 20.7, 21.0, 32.3]:
    n_excess = births * gap/100
    cost = n_excess * delta_2014
    rows_gap.append({"gap_pp": gap, "n_excess": int(n_excess),
                     "cost_egp_2014_prices": int(cost)})
    print(f"  {gap:>10.1f} {n_excess:>20,.0f} {'EGP ' + format(cost, ',.0f'):>26}")
print("""
  Read these as resource burden in 2014 prices, not as money that could be
  recovered. 12.8 pp is the figure the private-sector standardisation
  produced independently, and it sits inside the range the Robson
  standardisation produced from an entirely different method and data source.
""")

# =============================================================================
print("=" * 78)
print("3. WHY 2014 COSTS CANNOT SIMPLY BE UPRATED")
print("=" * 78)
print("""
  The Egyptian pound went from roughly 7 to the dollar in 2014 to roughly 48
  in 2025. Applying EGP 412 to a 2025 birth count would produce a figure
  that is nominally meaningless.

  The 2025 prices available are also prices, not costs, and they come from a
  press source rather than a costing study:

      public hospital caesarean ..... EGP 3,000-6,000
      private facility caesarean .... EGP 15,000-30,000

  No 2025 vaginal-delivery comparator was found, so no 2025 differential can
  be constructed. Reporting a 2025 saving figure would therefore require
  inventing the missing half of the subtraction.

  RECOMMENDATION: report the 2014 estimate as the sourced, defensible
  figure, present 2025 prices descriptively alongside it, and state plainly
  that a current differential cannot be computed until a current costing
  study exists. That absence is itself worth reporting — a country
  performing 1.35 million caesareans a year has not costed them since 2014.
""")

# =============================================================================
print("=" * 78)
print("4. WHAT THE COST STUDY FOUND BESIDES COSTS")
print("=" * 78)
print("""
  The same study asked physicians and key informants why caesarean use is
  high in Egypt. They unanimously agreed it is over-used, and named:

    financial incentive
    doctors' desire for control over their time
    fear of medical litigation
    vagueness of medical protocols
    limited opportunities for junior doctors to practise vaginal delivery
    shortage of pain-relief drugs in public hospitals
    shortage of anaesthetists trained in epidural administration

  Compare that list with the reasons Benha's authors gave independently for
  their 100% caesarean rate in Robson group 5: fear of litigation, no
  continuous fetal monitoring, no one-to-one care, no training in external
  cephalic version. Two separate sources, overlapping explanations, neither
  of them maternal preference.

  One further figure from the same study: only 6% of the women interviewed
  reported receiving any counselling on the disadvantages or health risks of
  a caesarean. Set that against the national survey finding that 80% of
  Egyptian women believe doctors prefer caesarean delivery.
""")

json.dump({"costs": COSTS, "published_2014": PUBLISHED_2014,
           "excess_share_scenarios": rows_2014,
           "gap_scenarios": rows_gap,
           "delta_2014_egp": delta_2014},
          open("egypt_cost_model.json","w"), indent=2)
c.to_csv("egypt_cost_sources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(rows_gap).to_csv("egypt_cost_by_gap.csv", index=False)
print("Saved egypt_cost_model.json, egypt_cost_sources.csv, egypt_cost_by_gap.csv")

### 4.2 — `egypt_governorates.py`: The Geographic Spread

**The trap this script exists to avoid:** Two governorate datasets circulate together and are routinely conflated:

- **EFHS 2021 (CAPMAS):** Precise per-governorate figures (Port Said 91.3%, Matrouh 39.3%)
- **MoH 2025:** NO per-governorate figures. Only bands ("three governorates exceeded 90%")

The three governorates named in 2025 are the same as those topping 2021, making it easy to attach 2021 numbers to 2025. **That would be wrong** — Matrouh was 39.3% in 2021 and "below 60%" in 2025, meaning it ROSE by up to 20 points.

**What cannot be done:** No governorate-level Robson distribution exists. The standardisation cannot run at governorate level. The governorate page is descriptive.

**What the spread supports:** A range of 39.3% to 91.3% inside one country, under one set of guidelines, cannot be explained by obstetric risk differences. It is variation in practice.

In [ ]:
"""
=============================================================================
EGYPT GOVERNORATE DATA
=============================================================================
    python egypt_governorates.py

THE TRAP THIS SCRIPT EXISTS TO AVOID

Two different governorate datasets circulate together and are routinely
conflated:

  Egypt Family Health Survey 2021 (CAPMAS)
      precise per-governorate figures — Port Said 91.3%, Kafr El-Sheikh
      88.4%, Gharbeya 84.3%, Matrouh 39.3%

  Ministry of Health birth registration, 2025 (announced 11 Jan 2026)
      NO per-governorate figures at all. Only bands: three governorates
      "exceeded 90%", two were "below 60%", national rate about 80%.

The three governorates named in the 2025 statement are the same three that
topped the 2021 survey, so it is easy to attach the 2021 numbers to the 2025
announcement. That would be wrong, and in one case badly wrong: Matrouh was
39.3% in 2021 and is described as below 60% in 2025, which means it ROSE by
up to twenty points. Quoting 39.3% as a current figure would understate it.

This file keeps the two sources in separate tables, with a `source` column
on every row, so they cannot be silently merged in a dashboard.

WHAT IS AND IS NOT AVAILABLE

Per-governorate caesarean rates exist for 2021 only. For 2025 there are
bands. No governorate-level Robson distribution exists for any year, so the
standardisation cannot be run at governorate level — the governorate page is
descriptive, and should say so.
=============================================================================
"""

import pandas as pd

# =============================================================================
# 2021 — Egypt Family Health Survey (CAPMAS). Precise figures.
# =============================================================================
GOV_2021 = [
    ("Port Said",      91.3, "urban",       "highest recorded"),
    ("Kafr El-Sheikh", 88.4, "Lower Egypt", ""),
    ("Gharbeya",       84.3, "Lower Egypt", ""),
    ("Damietta",       82.5, "Lower Egypt", "100% attended by a health provider"),
    ("Matrouh",        39.3, "frontier",    "lowest recorded"),
]

REGION_2021 = [
    ("Lower Egypt governorates", 60.3, 78.5, "+18.2"),
    ("Urban governorates",       62.0, 75.0, "+13.0"),
    ("Upper Egypt governorates", 39.7, 66.4, "+26.7"),
]

# =============================================================================
# 2025 — Ministry of Health birth registration. Bands only.
# =============================================================================
GOV_2025 = [
    ("Kafr El-Sheikh", "above 90%", "named in the ministerial statement"),
    ("Port Said",      "above 90%", "named in the ministerial statement"),
    ("Gharbeya",       "above 90%", "named in the ministerial statement"),
    ("Matrouh",        "below 60%", "named in the ministerial statement"),
    ("South Sinai",    "below 60%", "named in the ministerial statement"),
]

NATIONAL_2025 = {
    "rate": 80.0,
    "coverage": "more than 5,000 health facilities",
    "system": "electronic birth and death registration",
    "announced": "11 January 2026",
    "private_share_of_caesareans": 93.0,
}

# =============================================================================
print("=" * 74)
print("2021 — Egypt Family Health Survey, per governorate")
print("=" * 74)
g21 = pd.DataFrame(GOV_2021, columns=["governorate", "cs_rate_pct", "region", "note"])
g21["source"] = "Egypt Family Health Survey 2021 (CAPMAS)"
print(g21[["governorate", "cs_rate_pct", "region", "note"]].to_string(index=False))
print(f"\n  spread across named governorates: "
      f"{g21.cs_rate_pct.max() - g21.cs_rate_pct.min():.1f} percentage points")

print("\n" + "=" * 74)
print("2021 — regional trend, against 2014")
print("=" * 74)
r21 = pd.DataFrame(REGION_2021, columns=["region", "rate_2014", "rate_2021", "change_pp"])
r21["source"] = "Egypt Family Health Survey 2021 vs EDHS 2014"
print(r21[["region", "rate_2014", "rate_2021", "change_pp"]].to_string(index=False))
print("""
  Upper Egypt rose fastest — 26.7 points in seven years, from the lowest
  regional base. The gap between regions is closing upward rather than the
  high regions coming down, which is the opposite of what a successful
  containment policy would produce.
""")

print("=" * 74)
print("2025 — Ministry of Health registration data")
print("=" * 74)
g25 = pd.DataFrame(GOV_2025, columns=["governorate", "band", "note"])
g25["source"] = "MoH electronic birth registration 2025 (announced Jan 2026)"
print(g25[["governorate", "band"]].to_string(index=False))
print(f"""
  National rate {NATIONAL_2025['rate']:.0f}%, from {NATIONAL_2025['coverage']}.
  {NATIONAL_2025['private_share_of_caesareans']:.0f}% of these caesareans were in the private sector.

  NOTE: the 2025 release gives BANDS, not figures. Any dashboard showing a
  precise number against a governorate for 2025 is showing a 2021 number
  with the wrong year attached.
""")

print("=" * 74)
print("WHERE THE TWO SOURCES CAN BE COMPARED")
print("=" * 74)
comp = []
band_map = {"above 90%": (90, 100), "below 60%": (0, 60)}
for gov, band, _ in GOV_2025:
    row = g21[g21.governorate == gov]
    v21 = float(row.cs_rate_pct.iloc[0]) if len(row) else None
    lo, hi = band_map[band]
    if v21 is None:
        direction = "no 2021 figure available"
    elif v21 < lo:
        direction = f"ROSE — was {v21:.1f}% in 2021, now {band}"
    elif v21 > hi:
        direction = f"fell — was {v21:.1f}% in 2021, now {band}"
    else:
        direction = f"consistent — {v21:.1f}% in 2021, {band} in 2025"
    comp.append({"governorate": gov, "rate_2021": v21, "band_2025": band,
                 "reading": direction})
    print(f"  {gov:<16} {direction}")
print("""
  Matrouh is the one to be careful with. At 39.3% in 2021 and below 60% in
  2025, the only safe statement is that it rose. Quoting 39.3% as current
  would understate it by up to twenty points.
""")

print("=" * 74)
print("WHAT CANNOT BE DONE AT GOVERNORATE LEVEL")
print("=" * 74)
print("""
  No Robson distribution exists for any Egyptian governorate. The
  standardisation that produces the expected-versus-observed gap needs group
  sizes, and those exist only for Benha and Qena — two hospitals, not
  governorates.

  So the governorate page is descriptive: it shows the spread and its
  trajectory, and it does NOT show an excess-caesarean estimate by
  governorate. Presenting one would imply a calculation that has not been
  done.

  What the spread does support is the argument itself. A range this wide
  inside one country, under one set of clinical guidelines and one training
  system, cannot be explained by differences in obstetric risk between
  governorates. It is variation in practice, which is what the project set
  out to measure.
""")

for df, name in [(g21, "egypt_governorates_2021.csv"),
                 (r21, "egypt_regions_2021.csv"),
                 (g25, "egypt_governorates_2025_bands.csv"),
                 (pd.DataFrame(comp), "egypt_governorates_comparison.csv")]:
    df.to_csv(name, index=False, encoding="utf-8-sig")
    print(f"Saved {name}")

---

## Summary

### Data sources established
| Source | What it provides | Limitation |
|--------|-----------------|------------|
| DHS 2014 | National parity + previous CS structure | No presentation, onset, or gestational age |
| EHIS 2015 | National attitudes + lifetime CS measure | Not comparable to DHS CS rate (different denominator) |
| Benha 2018 | Complete Robson table, n=850 | Tertiary referral, not representative |
| Qena 2022-23 | Complete Robson table, n=2,186 | Same limitation |

### Corrections applied
| What | Old | Corrected | Effect |
|------|-----|-----------|--------|
| Category count | 10 | 8 (merged onset-of-labour groups) | Benha gap fell from +7.54 to +4.46 |
| Group 5 reallocation | Proportional to all groups | To Groups 3+4 only | Old method misplaced 6.6% of births |
| Group 9 reference | 46.52% (contaminated) | 100% (definitional) | ~0.2pp effect |
| Group 10 scenario | Built and used | **Withdrawn** | Cannot defend the reallocation |

### Key results
| Hospital | Gap A (current practice) | Gap B (total accumulated) | Inherited share |
|----------|------------------------|--------------------------|----------------|
| **Benha** | **+4.46 pp** | +22.01 pp | **80%** |
| **Qena** | **+20.65 pp** | +41.57 pp | **36%** |

### Next notebooks
- **Notebook 4** → Predictive model (XGBoost) — the other half of the project
- **Notebook 5** → SHAP interpretation & temporal validation
- **Notebook 6** → Export & packaging

---
*End of Notebook 3*